# Backend 1 데모 — Spatial covariant derivative $D_i$

ADM 3-slice 위에서의 공변 미분 $D_i$를 first-class로 다룬다.

이 노트북에서 검증할 것:
1. **3-slice 기본 셋업** — 3D `IndexSpace`와 3-metric $\Omega_{ij}$, 그리고 spatial Levi-Civita connection $\gamma^i{}_{jk}$
2. **$D_i$의 파싱·디스플레이** — `D_{i} V^{j}` LaTeX 입력 → `SpatialCovariantDeriv` 노드
3. **$\partial$ + $\gamma$ 전개**
4. **Variation 통합** — 배경 connection vs. varying connection (Palatini 3D)
5. **End-to-end** — RECALL 식의 $\delta g_{0i}$ 안의 $D_i B + \hat B_i$ 항을 IndexCalc으로 표현

## 1. 3-slice 셋업

In [1]:
from indexcalc import (
    IndexSpace, MetricRegistry, Tensor, LeviCivitaConnection,
    SpatialCovariantDeriv, spatial_covariant, expand_spatial_covariant,
    IndexRegistry, parse, to_latex,
    Variation, VariationRegistry, expand_variation,
)

spatial = IndexSpace('spatial', dim=3, indices='ijklmn', metric='Ω')

Om   = Tensor('Ω', [spatial.lower('i'), spatial.lower('j')])
Oinv = Tensor('Ω', [spatial.upper('i'), spatial.upper('j')])
metrics = MetricRegistry()
metrics.register(Om, Oinv, spatial)

gamma = LeviCivitaConnection(Om, Oinv, spatial)
print('3-Christoffel symbol definition:')
print(' ', to_latex(gamma.definition()))

3-Christoffel symbol definition:
  \frac{1}{2} \Omega^{i l} (\partial_{j} \Omega_{l k} + \partial_{k} \Omega_{l j} - \partial_{l} \Omega_{j k})


## 2. 파서·디스플레이 — `D_{i}` 토큰

파서는 `NAME='D'` 다음에 `LOWER` 토큰이 오면 spatial covariant 연산자로 해석.
(텐서 이름으로 `D`를 쓰는 케이스는 `\mathcal{D}` 등으로 회피.)

In [2]:
reg = IndexRegistry()
reg.register(spatial)

expr1 = parse(r'D_{i} \phi', reg)
print('parse(D_i φ) →', expr1)
print('to_latex     →', to_latex(expr1))

expr2 = parse(r'D_{i} V^{j}', reg)
print('\nparse(D_i V^j) →', expr2)
print('to_latex       →', to_latex(expr2))

# 2차 미분도 자연스럽게 nest
expr3 = parse(r'D_{i} D_{j} \phi', reg)
print('\nparse(D_i D_j φ) →', expr3)

parse(D_i φ) → D_i(φ)
to_latex     → D_{i} \phi

parse(D_i V^j) → D_i(V^j)
to_latex       → D_{i} V^{j}

parse(D_i D_j φ) → D_i(D_j(φ))


## 3. $\partial + \gamma$ 전개

$D_j V^i = \partial_j V^i + \gamma^i{}_{jk} V^k$

(IndexCalc에선 connection 기호가 기본값 `Γ`이지만 spatial 공간에서는 의미상 $\gamma$.)

In [3]:
V = Tensor('V', [spatial.upper('i')])
DV = SpatialCovariantDeriv(V, spatial.lower('j'), gamma)
expanded = expand_spatial_covariant(DV)
print('D_j V^i =', to_latex(expanded))

# Lower index 케이스: -Γ
W = Tensor('W', [spatial.lower('i')])
DW = SpatialCovariantDeriv(W, spatial.lower('j'), gamma)
print('D_j W_i =', to_latex(expand_spatial_covariant(DW)))

D_j V^i = \partial_{j} V^{i} + \Gamma^{i}{}_{j i_{1}} V^{i_{1}}
D_j W_i = \partial_{j} W_{i} - \Gamma^{i_{1}}{}_{j i} W_{i_{1}}


## 4. Variation 통합

### 4-1. 배경 connection ($\delta\gamma = 0$)

$\delta(D_i V^j) = D_i(\delta V^j)$

In [4]:
vreg = VariationRegistry()
vreg.declare_varying('V')

expr = Variation(SpatialCovariantDeriv(V, spatial.lower('i'), gamma))
result = expand_variation(expr, vreg)
print('δ(D_i V^j) =', to_latex(result))

δ(D_i V^j) = D_{i} δV^{i}


### 4-2. Varying connection ($\delta\gamma \neq 0$ — Palatini 3D)

$\delta(D_i V^j) = D_i(\delta V^j) + \delta\gamma^j{}_{ik} V^k$

In [5]:
vreg2 = VariationRegistry()
vreg2.declare_varying('V')
vreg2.declare_varying_connection('Γ')   # γ Christoffel은 IndexCalc 기본 기호 Γ

expr = Variation(SpatialCovariantDeriv(V, spatial.lower('i'), gamma))
result = expand_variation(expr, vreg2)
print('δ(D_i V^j) =', to_latex(result))

δ(D_i V^j) = D_{i} δV^{i} + δΓ^{i}{}_{i i_{1}} V^{i_{1}}


### 4-3. Lower index도 함께 — 부호 검증

$\delta(D_i T_{jk}) = D_i(\delta T_{jk}) - \delta\gamma^l{}_{ij} T_{lk} - \delta\gamma^l{}_{ik} T_{jl}$

In [6]:
T = Tensor('T', [spatial.lower('j'), spatial.lower('k')])
vreg3 = VariationRegistry()
vreg3.declare_varying('T')
vreg3.declare_varying_connection('Γ')

expr = Variation(SpatialCovariantDeriv(T, spatial.lower('i'), gamma))
print('δ(D_i T_{jk}) =', to_latex(expand_variation(expr, vreg3)))

δ(D_i T_{jk}) = D_{i} δT_{j k} + -δΓ^{i_{1}}{}_{i j} T_{i_{1} k} - δΓ^{i_{2}}{}_{i k} T_{j i_{2}}


## 5. End-to-end — RECALL 식 조각 표현

RECALL 식의 $\delta g_{0i}$ 안 항: $N(t)\,a(t)\,(D_i B + \hat B_i)$. 여기서 $B$는 scalar perturbation, $\hat B_i$는 transverse vector. 이걸 IndexCalc LaTeX로 직접 파싱·전개해본다.

(스칼라 prefactor `N a`는 일단 빼고 $D_i B + \hat B_i$ 부분만 다룬다.)

In [7]:
# B는 scalar이므로 D_i B 한 텀, B̂_i는 spatial vector
fragment = parse(r'D_{i} B + \hat{B}_{i}', reg)
print('파싱 결과 LaTeX:', to_latex(fragment))

# 전개
from indexcalc import expand_covariant
expanded = expand_covariant(fragment)
print('전개 후       :', to_latex(expanded))

# 변분: B와 B̂가 모두 perturbation이라면 사실 이미 1차 양인데,
# 데모 목적으로 한 번 더 δ를 걸어보자 (2차 변분)
vreg = VariationRegistry()
vreg.declare_varying('B')
vreg.declare_varying('B̂')
second_var = expand_variation(Variation(fragment), vreg)
print('δ 적용         :', to_latex(second_var))

파싱 결과 LaTeX: D_{i} B + \hat{B}_{i}
전개 후       : \partial_{i} B + \hat{B}_{i}
δ 적용         : D_{i} δB + \hat{δB}_{i}


## 정리

| 항목 | 상태 |
|---|---|
| 3D `IndexSpace` + Ω metric raise/lower | ✅ 기존 코드로 직접 동작 |
| 3-Christoffel 자동 생성 (`LeviCivitaConnection`) | ✅ 기존 |
| `SpatialCovariantDeriv` 클래스 | ✅ 신규 (`CovariantDeriv` subclass) |
| `D_{i}` LaTeX 파싱 | ✅ 신규 |
| `D_i` LaTeX 출력 | ✅ 신규 |
| `expand_spatial_covariant` (∂+γ 전개) | ✅ 기존 expand 재사용 |
| Variation Leibniz + 3D Palatini ($\delta\gamma$) | ✅ 신규 (기존 Palatini 메커니즘 재사용) |

**남은 항목** (Backend 1 범위 밖):
- ADM full decomposition (lapse $N$, shift $N^i$, extrinsic curvature $K_{ij}$, projector $h_{\mu\nu}$)
- 4D ↔ 3D 인덱스 자동 변환 (현재는 사용자가 spatial IndexSpace를 명시적으로 사용)
- Vielbein identity 자동 인식

이들은 별도 Backend 1.b / Backend 2 로드맵.